In [6]:
import mlflow
import pandas as pd
import mlflow.sklearn
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import pandas as pd
import re
import string
%pip install nltk

import nltk

nltk.download("stopwords")
nltk.download("wordnet")
nltk.download("omw-1.4")

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import numpy as np

  Using cached defusedxml-0.7.1-py2.py3-none-any.whl.metadata (32 kB)
  Using cached tqdm-4.70.1-py3-none-any.whl.metadata (57 kB)
   ---------------------------------------- 0.0/1.8 MB ? eta -:--:--
   ----------------------- ---------------- 1.0/1.8 MB 7.2 MB/s eta 0:00:01
   ---------------------------------------- 1.8/1.8 MB 7.0 MB/s  0:00:00
Using cached defusedxml-0.7.1-py2.py3-none-any.whl (25 kB)
Using cached tqdm-4.70.1-py3-none-any.whl (80 kB)

   ---------------------------------------- 0/4 [tqdm]
   ---------------------------------------- 0/4 [tqdm]
   ---------------------------------------- 0/4 [tqdm]
   ---------------------------------------- 0/4 [tqdm]
   ---------------------------------------- 0/4 [tqdm]
   ---------- ----------------------------- 1/4 [regex]
   ---------- ----------------------------- 1/4 [regex]
   -------------------- ------------------- 2/4 [defusedxml]
   ------------------------------ --------- 3/4 [nltk]
   ------------------------------ ----

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\aanam\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\aanam\AppData\Roaming\nltk_data...
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\aanam\AppData\Roaming\nltk_data...


In [7]:
df = pd.read_csv("IMDB.csv")
df = df.sample(500)
df.to_csv("data.csv", index=False)
df.head()

,review,sentiment
975,"In this unlikely love triangle, set in 19th ce...",positive
604,What starts off fairly well (and quite disturb...,negative
636,"I'm guessing that we all, no matter if we are ...",positive
169,While this movie did have a few scary moments ...,negative
584,Faith and Mortality... viewed through the lens...,positive


In [8]:
# data preprocessing
# Define text preprocessing functions
def lemmatization(text):
    """Lemmatize the text."""
    lemmatizer = WordNetLemmatizer()
    text = text.split()
    text = [lemmatizer.lemmatize(word) for word in text]
    return " ".join(text)


def remove_stop_words(text):
    """Remove stop words from the text."""
    stop_words = set(stopwords.words("english"))
    text = [word for word in str(text).split() if word not in stop_words]
    return " ".join(text)


def removing_numbers(text):
    """Remove numbers from the text."""
    text = "".join([char for char in text if not char.isdigit()])
    return text


def lower_case(text):
    """Convert text to lower case."""
    text = text.split()
    text = [word.lower() for word in text]
    return " ".join(text)


def removing_punctuations(text):
    """Remove punctuations from the text."""
    text = re.sub("[%s]" % re.escape(string.punctuation), " ", text)
    text = text.replace("؛", "")
    text = re.sub("\s+", " ", text).strip()
    return text


def removing_urls(text):
    """Remove URLs from the text."""
    url_pattern = re.compile(r"https?://\S+|www\.\S+")
    return url_pattern.sub(r"", text)


def normalize_text(df):
    """Normalize the text data."""
    try:
        df["review"] = df["review"].apply(lower_case)
        df["review"] = df["review"].apply(remove_stop_words)
        df["review"] = df["review"].apply(removing_numbers)
        df["review"] = df["review"].apply(removing_punctuations)
        df["review"] = df["review"].apply(removing_urls)
        df["review"] = df["review"].apply(lemmatization)
        return df
    except Exception as e:
        print(f"Error during text normalization: {e}")
        raise

In [9]:
df = normalize_text(df)
df.head()

,review,sentiment
975,unlikely love triangle set th century italy th...,positive
604,start fairly well and quite disturbing quickly...,negative
636,guessing all matter fan car luv sound dodge ch...,positive
169,movie scary moment great use music film angle ...,negative
584,faith mortality viewed lens elderly ashkenazi ...,positive


In [10]:
df["sentiment"].value_counts()

sentiment
positive    253
negative    247
Name: count, dtype: int64

In [11]:
x = df["sentiment"].isin(["positive", "negative"])
df = df[x]

In [12]:
df["sentiment"] = df["sentiment"].map({"positive": 1, "negative": 0})
df.head()

,review,sentiment
975,unlikely love triangle set th century italy th...,1
604,start fairly well and quite disturbing quickly...,0
636,guessing all matter fan car luv sound dodge ch...,1
169,movie scary moment great use music film angle ...,0
584,faith mortality viewed lens elderly ashkenazi ...,1


In [13]:
df.isnull().sum()

review       0
sentiment    0
dtype: int64

In [20]:
vectorizer = CountVectorizer(max_features=50)
X = vectorizer.fit_transform(df["review"])
y = df["sentiment"]

In [21]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [18]:
import dagshub

mlflow.set_tracking_uri(
    "https://dagshub.com/swati-mishra07/MLOPS-Capstone-Project.mlflow"
)
dagshub.init(repo_owner='swati-mishra07', repo_name='MLOPS-Capstone-Project', mlflow=True)

# mlflow.set_experiment("Logistic Regression Baseline")
mlflow.set_experiment("Logistic Regression Baseline")


❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

c:\Users\aanam\OneDrive\Documents\MLOPS\MLOPS-Capstone-Project\atlas\Lib\site-packages\rich\live.py:260: 
UserWarning: install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=335e8c63-c5b0-448d-aa00-e66787c3f8e4&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=0a7bea54d451f96d10c4a00cc5e79216f2239945291bcfc059232c5d2a53063a




Accessing as swati-mishra07

Initialized MLflow to track repo "swati-mishra07/MLOPS-Capstone-Project"

Repository swati-mishra07/MLOPS-Capstone-Project initialized!

2026/09/19 16:43:58 INFO mlflow.tracking.fluent: Experiment with name 'Logistic Regression Baseline' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/9952c88e37fa4ad28b9e68b093b641b7', creation_time=1789816437304, effective_trace_archival_retention=None, experiment_id='0', last_update_time=1789816437304, lifecycle_stage='active', name='Logistic Regression Baseline', tags={}, trace_location=None, workspace='default'>

In [22]:
import mlflow
import logging
import os
import time
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Configure logging
logging.basicConfig(
    level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s"
)

logging.info("Starting MLflow run...")

with mlflow.start_run():
    start_time = time.time()

    try:
        logging.info("Logging preprocessing parameters...")
        mlflow.log_param("vectorizer", "Bag of Words")
        mlflow.log_param("num_features", 50)
        mlflow.log_param("test_size", 0.2)

        logging.info("Initializing Logistic Regression model...")
        model = LogisticRegression(
            max_iter=1000
        )  # Increase max_iter to prevent non-convergence issues

        logging.info("Fitting the model...")
        model.fit(X_train, y_train)
        logging.info("Model training complete.")

        logging.info("Logging model parameters...")
        mlflow.log_param("model", "Logistic Regression")

        logging.info("Making predictions...")
        y_pred = model.predict(X_test)

        logging.info("Calculating evaluation metrics...")
        accuracy = accuracy_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred)
        recall = recall_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred)

        logging.info("Logging evaluation metrics...")
        mlflow.log_metric("accuracy", accuracy)
        mlflow.log_metric("precision", precision)
        mlflow.log_metric("recall", recall)
        mlflow.log_metric("f1_score", f1)

        logging.info("Saving and logging the model...")
        mlflow.sklearn.log_model(model, "model")

        # Log execution time
        end_time = time.time()
        logging.info(
            f"Model training and logging completed in {end_time - start_time:.2f} seconds."
        )

        # Save and log the notebook
        # notebook_path = "exp1_baseline_model.ipynb"
        # logging.info("Executing Jupyter Notebook. This may take a while...")
        # os.system(f"jupyter nbconvert --to notebook --execute --inplace {notebook_path}")
        # mlflow.log_artifact(notebook_path)

        # logging.info("Notebook execution and logging complete.")

        # Print the results for verification
        logging.info(f"Accuracy: {accuracy}")
        logging.info(f"Precision: {precision}")
        logging.info(f"Recall: {recall}")
        logging.info(f"F1 Score: {f1}")

    except Exception as e:
        logging.error(f"An error occurred: {e}", exc_info=True)

2026-09-19 16:53:04,093 - INFO - Starting MLflow run...
2026-09-19 16:53:05,944 - INFO - Logging preprocessing parameters...
2026-09-19 16:53:07,166 - INFO - Initializing Logistic Regression model...
2026-09-19 16:53:07,167 - INFO - Fitting the model...
2026-09-19 16:53:07,187 - INFO - Model training complete.
2026-09-19 16:53:07,188 - INFO - Logging model parameters...
2026-09-19 16:53:07,676 - INFO - Making predictions...
2026-09-19 16:53:07,678 - INFO - Calculating evaluation metrics...
2026-09-19 16:53:07,694 - INFO - Logging evaluation metrics...
2026-09-19 16:53:09,118 - INFO - Saving and logging the model...
2026/09/19 16:53:09 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026-09-19 16:53:29,716 - INFO - Model training and logging completed in 23.77 seconds.
2026-09-19 16:53:29,717 - INFO - Accuracy: 0.6
2026-09-19 16:53:29,718 - INFO - Precision: 0.68
2026-09-19 16:53:29,719 - INFO - Recall: 0.5862068965517241
2026-09-19 16:53:29,720 - 

🏃 View run awesome-finch-740 at: https://dagshub.com/swati-mishra07/MLOPS-Capstone-Project.mlflow/#/experiments/0/runs/67d312b646c345a98ff3d98a8dc5fea8
🧪 View experiment at: https://dagshub.com/swati-mishra07/MLOPS-Capstone-Project.mlflow/#/experiments/0
